### Lösung zu Sokoban finden
Wir wollen breadth-first search nutzen, um eine Lösun zu folgendem Sokoban Level zu finden.  

<img src='/files/images/sokoban_level1.png'>

Das Sokoban Spielfeld denken wir uns als 8x8 Gitter.  
Der Suchraum besteht aus allen Paaren (Boxpositionen, Spielerposition), wobei wir 
zwei solche Paare `(boxes_1, spieler_1)` und `(boxes_2, spieler_2)` als gleich ansehen, falls
- `boxes_1` und `boxes_2` Mengen mit den gleichen Elementen sind, und
- der Spieler von  Position `spieler_1` nach `spieler_2` ziehen kann, ohne Boxen verschieben zu müssen. 


Im Modul `sokoban_graph` ist ein Graph (in der Tat ein Baum)  mit allen möglichen Paaren `(boxes, spieler)` gespeichert. Das heisst, das von der Startposition aus einen Weg zu jeder anderen Position gibt, dass aber
nicht alle Nachbarn im Dict `graph` aufgelistet sind. Insbesondere ist der Graph kreisfrei.

Um den Graphen schlank zu halten, haben wir jedem Spielfeld einen Index zugeordnet.
Diese Zuordnung ist im Dict `pos_id` gespeichert, die umgekehrte Zuordnung ist im Dict `id_pos`.

Da eine Menge nicht als Schlüssel in einem Dict auftreten können, verwenden wir
ein `frozenset`, eine immutable (unveränderbare) Menge.
Beachte, dass im Gegensatz zu Tuple und Listen, folgender Vergleich `True` liefert.
```
frozenset({1, 2}) == {2, 1}
```

In [ ]:
frozenset({1, 2}) == {2, 1}, (1, 2) == [1, 2]

In [ ]:
import sokoban_graph as S


help(S)

In [ ]:
import sokoban_graph as S
from collections import deque


def search_bf(start, graph, target=None):
    node_to_visite = deque([start])
    go_back = {start: None}

    while node_to_visite:
        node = node_to_visite.pop()
        if target and node[0] == target:
            return node, go_back

        for neighbor in graph.get(node, ()):
            if neighbor in go_back:
                continue
            go_back[neighbor] = node
            node_to_visite.appendleft(neighbor)


def get_path_home(node, go_back):
    path = []
    while node:
        path.append(node)
        node = go_back[node]

    return path

In [ ]:
goal, child_parent = search_bf((S.boxes, S.player), S.graph, S.targets)

In [ ]:
goal

In [ ]:
path = get_path_home(goal, child_parent)
path

In [ ]:
path_to_target = [x[0] for x in reversed(path)]
path_to_target

In [ ]:
# welche Kiste wurde bewegt?
a, b = path_to_target[:2]
a, b, a - b, set((a - b)).pop()

In [ ]:
def get_moves(path):
    moves = []
    for a, b in zip(path[:-1], path[1:]):
        x, y = set((a - b)).pop(), set((b - a)).pop()
        moves.append((S.id_pos[x], S.id_pos[y]))
    return moves

In [ ]:
moves = get_moves(path_to_target)
moves